In [1]:
import sys
sys.executable

'/workspaces/LLM_workspace/02_vector_search/.venv/bin/python'

# Module 2 Homework: Vector Search

Link to the github repo: [Link](https://github.com/DataTalksClub/llm-zoomcamp/tree/main)

## Q1. Embedding a query

Embed the following query:

In [2]:
from embedder import Embedder
query = "How does approximate nearest neighbor search work?"
embed = Embedder()
q1 = embed.encode(query)
q1[0]

2026-06-22 21:04:39.354074169 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


np.float64(-0.02058203437252893)

## Loading the data

Let's pull the lesson pages from the repository:

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
len(documents)
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Cosine similarity

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its content, and compute the cosine similarity with the query vector from Q1.

In [5]:
page = "02-vector-search/lessons/07-sqlitesearch-vector.md"
document = next((doc for doc in documents if doc["filename"] == page), None)

doc_encode = embed.encode(document["content"])
doc_encode.dot(q1)

np.float64(0.36107027225589694)

## Q3. Chunking and search by hand
Let's chunk the documents to split the topics at each document and then find the highest similarity chunk with q1

In [6]:
from gitsource import chunk_documents
from tqdm import tqdm
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

X = []
for i in tqdm(range(len(chunks))):
    chunk = chunks[i]
    batch_encode = embed.encode(chunk["content"])
    X.append(batch_encode)

X = np.array(X)

scores = X.dot(q1)
idx = np.argmax(scores)
chunks[idx]

  0%|          | 0/295 [00:00<?, ?it/s]

100%|██████████| 295/295 [00:19<00:00, 15.02it/s]


{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

## Q4. Vector search with minsearch

In [7]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)


query = 'What metric do we use to evaluate a search engine?'
q2 = embed.encode(query)
results = vindex.search(q2, num_results=1)

results[0]


{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

## Q5. Text search vs vector search

Let's compare the text search with vector search:

In [11]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(chunks)

query = "How do I store vectors in PostgreSQL?"
q3 = embed.encode(query)

r_vs = vindex.search(q3, num_results=5)
r_ts = index.search(query, num_results=5)

r_vs_filenames = [r["filename"] for r in r_vs]
r_ts_filenames = [r["filename"] for r in r_ts]
print("Vector search results:", r_vs_filenames)
print("Text search results:", r_ts_filenames)


Vector search results: ['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']
Text search results: ['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']


## Q6. Hybrid search

In [12]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query = "How do I give the model access to tools?"
q4 = embed.encode(query)

r_vs = vindex.search(q4, num_results=5)
r_ts = index.search(query, num_results=5)

results = rrf([r_vs, r_ts])

results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'

# [Optional] Implementing with LangChain

In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from embedder import Embedder

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
chunks = chunk_documents(documents, size=2000, step=1000)

len(documents), len(chunks)

embed = Embedder()

We convert all of the inputs to langchain Document

In [4]:
from langchain_core.documents import Document

lc_docs = [
        Document(
            page_content=chunk["content"],
            metadata={
                "filename": chunk["filename"],
                "start": chunk.get("start", None),
            }
        ) for chunk in chunks
    ]
len(lc_docs)

295

In [5]:
lc_docs[0]

Document(metadata={'filename': '01-agentic-rag/lessons/01-intro.md', 'start': 0}, page_content='# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as th

Now its time to define our embedding, (it is also possible to use any other embedding methods as well such as OpenAIEmbedding using langchain.embeddings.openai). I just wanted to makesure that Chroma uses the same embedding as we used during the module 2 HW

In [6]:
from langchain_core.embeddings import Embeddings

class ONNXMiniLMEmbeddings(Embeddings):
    def __init__(self, embedder):
        self.embedder = embedder

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        vectors = self.embedder.encode_batch(texts)
        return [v.tolist() for v in vectors]

    def embed_query(self, text: str) -> list[float]:
        vector = self.embedder.encode(text)
        return vector.tolist()

embedding = ONNXMiniLMEmbeddings(embed)

### VectorSearch using Langchain chroma

Now let's define the vector search usinng Chroma through langchain

In [7]:
from langchain_chroma import Chroma

vector_db = Chroma.from_documents(
    documents=lc_docs,
    embedding=embedding,
    collection_name="llm_zoomcamp_module_2",
    persist_directory="./chroma_db",
)

In [8]:
query = "What metric do we use to evaluate a search engine?"

vector_results_with_scores = vector_db.similarity_search_with_score(query, k=5)

for doc, score in vector_results_with_scores:
    print(score, doc.metadata["filename"])

0.8028466105461121 04-evaluation/lessons/05-search-metrics.md
0.9286495447158813 04-evaluation/lessons/01-intro.md
1.0272178649902344 01-agentic-rag/lessons/05-search.md
1.0333185195922852 04-evaluation/lessons/01-intro.md
1.0573900938034058 04-evaluation/lessons/15-next-steps.md


the distance measure to check the similarity in langchain Chroma api by default is `L2`, that is why the numbers are higher than 1


### Text search using langchain BM25 

In [18]:
from langchain_community.retrievers import BM25Retriever
import re

def preprocess(text: str):
    return re.findall(r"(?u)\b\w\w+\b", text.lower())

keyword_retriever = BM25Retriever.from_documents(
    lc_docs,
    preprocess_func=preprocess)
keyword_retriever.k = 5

In [19]:
query = "How do I store vectors in PostgreSQL?"

keyword_results = keyword_retriever.invoke(query)

for i, doc in enumerate(keyword_results, start=1):
    print(i, doc.metadata["filename"])
    print(doc.page_content[:300])
    print("-" * 80)

1 02-vector-search/lessons/01-intro.md
dding model produces these vectors. It's a neural network
trained to capture meaning, so texts that mean similar things land on
similar vectors. We measure how close two vectors are with a distance
metric. The most common one is cosine similarity.

Cosine similarity measures the angle between two ve
--------------------------------------------------------------------------------
2 02-vector-search/lessons/01-intro.md
# Vector Search

Video: [Watch this lesson](https://www.youtube.com/watch?v=qyZgxTmC2cY&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In module 1 we used keyword search with minsearch and sqlitesearch.
It matches exact words. If you search for "Docker", the document has
to contain "Docker" to come back.
--------------------------------------------------------------------------------
3 03-orchestration/lessons/05-rag.md
arch](../../02-vector-search/lessons/04-vector-search.md).

## How RAG Works in Kestra

RAG has two phases. In the demo f

I guess the difference in answers for minsearch index and BM25Retriever is in the text scoring mechanism both methods use:
1. minsearch.index uses TF-IDF + cosine similarity for text fields
2. BM25Retriver is a different kinda TF-IDF mechanism

In [13]:
query = "How do I store vectors in PostgreSQL?"

vector_results = vector_db.similarity_search(query, k=5)
keyword_results = keyword_retriever.invoke(query)

print("VECTOR RESULTS")
for doc in vector_results:
    print(doc.metadata["filename"])

print("\nKEYWORD RESULTS")
for doc in keyword_results:
    print(doc.metadata["filename"])

VECTOR RESULTS
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md

KEYWORD RESULTS
03-orchestration/lessons/05-rag.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
01-agentic-rag/lessons/13-function-calling.md
02-vector-search/lessons/02-embeddings.md


### Hybrid search with LangChain

In [15]:
from langchain_classic.retrievers import EnsembleRetriever

In [20]:
vector_retriever = vector_db.as_retriever(
    search_kwargs={"k": 5}
)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, keyword_retriever],
    weights=[0.5, 0.5],
)

In [21]:
query = "How do I give the model access to tools?"

hybrid_results = hybrid_retriever.invoke(query)

for i, doc in enumerate(hybrid_results[:5], start=1):
    print(i, doc.metadata["filename"])
    print(doc.page_content[:300])
    print("-" * 80)

1 01-agentic-rag/lessons/01-intro.md
wrong.

## The project

RAG solves these problems by giving the LLM relevant documents at
question time. We don't hope the model memorized the answer. We
retrieve the right information and hand it to the LLM, and the model
generates a grounded response. This lets us inject knowledge the model
never 
--------------------------------------------------------------------------------
2 01-agentic-rag/lessons/13-function-calling.md
# Function Calling

Video: [Watch this lesson](https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson we built a RAG pipeline with `RAGBase.rag()`
and saw it fail on the "Olama" typo. The search returned nothing
useful, and the LLM had no way to 
--------------------------------------------------------------------------------
3 04-evaluation/lessons/02-ground-truth.md
swer.

## Loading the documents

We'll use helper files from module 01 and this module.

If you don't have t